# boolean-mask-identity-replace — worked example 1: Clip out-of-range entries to a bound (non-destructive)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `boolean-mask-identity-replace`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A comparison such as `x > hi` produces a `dtype=bool` mask with the same shape as `x`. Writing through that mask with `y[mask] = value` overwrites exactly the flagged entries; a scalar `value` broadcasts over the whole selected region. To keep the function non-mutating, clone first so the caller's input is never clobbered.

## Worked solution

We want every entry above `hi` set to `hi`, and every entry below `lo` set to `lo`, returning a fresh tensor.

1. **Clone.** `y = x.clone()` gives an independent copy so writing through it leaves `x` untouched. This is the single most common bug in masked-write code.
2. **Build the high mask and write.** `y > hi` is a bool tensor of the same shape flagging the too-large entries. `y[y > hi] = hi` selects exactly those positions and assigns the scalar `hi`, which broadcasts to fill them.
3. **Build the low mask and write.** `y < lo` flags the too-small entries; `y[y < lo] = lo` clamps them. We evaluate the masks against `y` (the working copy) rather than `x`, but since we only overwrote out-of-range values with in-range bounds, either source gives the same mask here.
4. **Return `y`.** The original `x` is unchanged because every mutation went through the clone.

In [ ]:
def clip_to_bounds(x: Tensor, lo: float, hi: float) -> Tensor:
    y = x.clone()
    y[y > hi] = hi
    y[y < lo] = lo
    return y

t.manual_seed(0)
x = t.randn(2, 4)
out = clip_to_bounds(x, -0.5, 0.5)
print('max <= 0.5 :', bool(out.max() <= 0.5))
print('min >= -0.5:', bool(out.min() >= -0.5))
print('x untouched:', bool(t.equal(x, x.clone()) and out.shape == x.shape))